# SEPTA Regional Rail — World Cup 2026 Optimization
## OIDD 2210 Final Project — End-to-End Demo

This notebook walks through the full pipeline:
1. Load real SEPTA ridership data + World Cup demand overlay
2. Run upper-level optimization (SLSQP)
3. Run bilevel optimization (iterative best-response)
4. Visualize results

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Demand

In [ ]:
from data.demand import get_total_demand
from data.network import LINES
from data.parameters import TIME_SLOTS, slot_label

demand_wc = get_total_demand(worldcup=True)
demand_base = get_total_demand(worldcup=False)

total_wc   = sum(demand_wc[l].sum()   for l in demand_wc)
total_base = sum(demand_base[l].sum() for l in demand_base)

print(f'Base demand (normal day):  {total_base:>10,.0f} passengers')
print(f'World Cup demand (match):  {total_wc:>10,.0f} passengers')
print(f'World Cup uplift:          {(total_wc/total_base - 1)*100:.1f}%')

In [ ]:
from utils.visualize import plot_demand_curve

# Show demand for the 4 highest-ridership lines
top4 = sorted(demand_wc, key=lambda l: demand_wc[l].sum(), reverse=True)[:4]
plot_demand_curve(demand_wc, lines=top4, worldcup=True)

## 2. Upper-Level Optimization (SLSQP continuous relaxation)

In [ ]:
from models.upper_level import solve as upper_solve, LNAMES, TBLOCKS, TBLOCK_RANGES

result_upper = upper_solve(demand_wc)

print(f"Profit:     ${result_upper['profit']:>12,.0f}")
print(f"Revenue:    ${result_upper['revenue']:>12,.0f}")
print(f"Fixed cost: ${result_upper['fixed_cost']:>12,.0f}")
print(f"Var cost:   ${result_upper['var_cost']:>12,.0f}")
print(f"Total pax:  {result_upper['total_pax']:>13,.0f}")
print(f"Equity OK:  {result_upper['equity_all']}")

In [ ]:
from utils.visualize import plot_allocation_heatmap, plot_fare_profile

plot_allocation_heatmap(result_upper)

In [ ]:
plot_fare_profile(result_upper, lines=top4)

## 3. Bilevel Optimization (iterative best-response)

In [ ]:
from models.bilevel import run_bilevel

result_bilevel = run_bilevel(demand_wc, max_iter=20, verbose=True)

print(f"\nConverged: {result_bilevel['converged']}")
print(f"Profit:    ${result_bilevel['profit']:>12,.0f}")
print(f"Total pax: {result_bilevel['total_pax']:>13,.0f}")

In [ ]:
from utils.visualize import plot_profit_convergence

plot_profit_convergence(result_bilevel['iterations'])

## 4. Per-Block Summary Table

In [ ]:
import pandas as pd

rows = []
for l in LNAMES:
    lr = result_bilevel['lines'][l]
    for blk in TBLOCKS:
        idxs = list(TBLOCK_RANGES[blk])
        total_f  = lr['f'][idxs].sum()
        total_x  = lr['x'][idxs].sum()
        avg_p    = float(np.average(lr['p'][idxs], weights=np.maximum(lr['x'][idxs], 1e-6)))
        avg_util = float(np.mean(lr['util'][idxs]))
        eq_ok    = bool(np.all(lr['equity_ok'][idxs]))
        rows.append({'line': l, 'block': blk, 'trains': round(total_f, 1),
                     'avg_fare': round(avg_p, 2), 'pax': round(total_x),
                     'utilization': round(avg_util, 3), 'equity_ok': eq_ok})

df = pd.DataFrame(rows)
df.style.format({'avg_fare': '${:.2f}', 'utilization': '{:.1%}'})

## 5. Compare: Normal Day vs World Cup

In [ ]:
result_base = upper_solve(demand_base)

print(f"{'Metric':<20} {'Normal day':>14} {'World Cup':>14} {'Delta':>12}")
print("-" * 62)
for key, label in [('profit','Profit ($)'), ('revenue','Revenue ($)'),
                    ('total_pax','Total pax')]:
    b, w = result_base[key], result_upper[key]
    print(f"{label:<20} {b:>14,.0f} {w:>14,.0f} {w-b:>+12,.0f}")